# PersuasiX — Model Training

Train the detection model (RoBERTa) and generation models (FLAN-T5) step by step.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from torch.utils.data import DataLoader

from src.utils.helpers import load_config, set_seed, get_device, count_parameters

cfg = load_config('../config/config.yaml')
set_seed(cfg['project']['seed'])
device = get_device()
print(f'Device: {device}')

## 1. Build Dataset

In [ ]:
from src.data.collector import DataCollector
from src.data.cleaner import DataCleaner
from src.data.enricher import LLMEnricher
from src.data.validator import DataValidator

# Stage 1: Collect
collector = DataCollector(output_dir='../data/raw')
raw_df = collector.collect_all()
print(f'Collected: {len(raw_df)} rows')

# Stage 2: Clean
cleaner = DataCleaner()
clean_df = cleaner.clean(raw_df)

# Stage 3: Enrich (fallback mode — no API key needed)
enricher = LLMEnricher(provider='none')
enricher._client = None
enriched_df = enricher.enrich(clean_df)

# Stage 4: Validate
validator = DataValidator(use_embeddings=False)
final_df = validator.validate(enriched_df)
print(f'Final dataset: {len(final_df)} rows')

## 2. Train Detector (RoBERTa)

In [ ]:
from src.models.detector import PersuasionDetector
from src.data.dataset import PersuasixDataset
from src.training.trainer import PersuasixTrainer

# Initialize model
model = PersuasionDetector(
    model_name=cfg['detector']['model_name'],
    num_labels=cfg['detector']['num_labels'],
    dropout=cfg['detector']['dropout'],
)
tokenizer = PersuasionDetector.get_tokenizer(cfg['detector']['model_name'])

params = count_parameters(model)
print(f"Trainable: {params['trainable_millions']}M / Total: {params['total_millions']}M")

# Create datasets
train_size = int(len(final_df) * 0.8)
train_df = final_df.iloc[:train_size]
val_df = final_df.iloc[train_size:]

train_ds = PersuasixDataset(train_df, tokenizer, max_length=128)
val_ds = PersuasixDataset(val_df, tokenizer, max_length=128)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=4)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

In [ ]:
# Train
trainer = PersuasixTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=3,
    learning_rate=2e-5,
    fp16=False,
    output_dir='../checkpoints/detector',
    task_type='classification',
    device=device,
)

history = trainer.train()

In [ ]:
from src.utils.visualization import plot_training_curves
fig = plot_training_curves(history)
import matplotlib.pyplot as plt
plt.show()

## 3. Quick Inference Test

In [ ]:
from src.data.collector import TECHNIQUE_LABELS

test_text = "Only a fool would disagree. Everyone knows this is the only solution!"
inputs = tokenizer(test_text, return_tensors='pt', max_length=128, truncation=True, padding='max_length')

model.eval()
model.to('cpu')
result = model.predict(inputs['input_ids'], inputs['attention_mask'], threshold=0.3)

probs = result['probabilities'][0]
for label, prob in sorted(zip(TECHNIQUE_LABELS, probs.tolist()), key=lambda x: -x[1])[:5]:
    print(f'{label:30s} {prob:.4f}')